<a href="https://colab.research.google.com/github/FarhadNuri/BN-Fact-Check/blob/main/notebooks/Day1_Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE = '/content/drive/MyDrive/BanglaFakeNews'
print("Base path set:", BASE)

Base path set: /content/drive/MyDrive/BanglaFakeNews


In [ ]:
import os

folders = [
    f'{BASE}/data/raw',
    f'{BASE}/data/processed',
    f'{BASE}/notebooks',
    f'{BASE}/models',
    f'{BASE}/app/templates',
    f'{BASE}/app/static',
    f'{BASE}/reports/figures',
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f'✅ Created: {folder}')

print('\nAll folders ready.')

✅ Created: /content/drive/MyDrive/BanglaFakeNews/data/raw
✅ Created: /content/drive/MyDrive/BanglaFakeNews/data/processed
✅ Created: /content/drive/MyDrive/BanglaFakeNews/notebooks
✅ Created: /content/drive/MyDrive/BanglaFakeNews/models
✅ Created: /content/drive/MyDrive/BanglaFakeNews/app/templates
✅ Created: /content/drive/MyDrive/BanglaFakeNews/app/static
✅ Created: /content/drive/MyDrive/BanglaFakeNews/reports/figures

All folders ready.


In [ ]:
# Fix httpx version conflict completely
!pip install httpx==0.27.0 -q
!pip install huggingface_hub --upgrade -q
!pip install transformers --upgrade -q
!pip install bnlp-toolkit -q
!pip install imbalanced-learn -q
!pip install deep-translator -q

print("✅ Done. Now restart runtime.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 7.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.0 which is incompatible.
mcp 1.26.0 requires httpx>=0.27.1, but you have httpx 0.27.0 which is incompatible.
google-genai 1.66.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 63.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 11.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.4/175.4 kB 18.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [ ]:
!pip install transformers -q
print("✅ Transformers installed.")

✅ Transformers installed.


In [ ]:


import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from transformers import AutoTokenizer

print("✅ All imports successful.")
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU not found — go to Runtime → Change runtime type → T4 GPU")

✅ All imports successful.
PyTorch version: 2.10.0+cu128
GPU available: True
GPU name: Tesla T4


In [ ]:
# Run this cell to upload train.csv and validate.csv from your computer
from google.colab import files

print("Select train.csv when the upload dialog opens...")
uploaded = files.upload()

for filename, content in uploaded.items():
    dest = f'{BASE}/data/raw/{filename}'
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'✅ Saved {filename} → {dest}')

Select train.csv when the upload dialog opens...


Saving train.csv to train (4).csv
✅ Saved train (4).csv → /content/drive/MyDrive/BanglaFakeNews/data/raw/train (4).csv


In [ ]:
# Run this cell to upload train.csv and validate.csv from your computer
from google.colab import files

print("Select train.csv when the upload dialog opens...")
uploaded = files.upload()

for filename, content in uploaded.items():
    dest = f'{BASE}/data/raw/{filename}'
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'✅ Saved {filename} → {dest}')

Select train.csv when the upload dialog opens...


Saving train.csv to train.csv
✅ Saved train.csv → /content/drive/MyDrive/BanglaFakeNews/data/raw/train.csv


In [ ]:
train = pd.read_csv(f'{BASE}/data/raw/train.csv')
validate = pd.read_csv(f'{BASE}/data/raw/validate.csv')

print("=== TRAIN ===")
print(f"Shape: {train.shape}")
print(f"Columns: {train.columns.tolist()}")
print(f"Label counts:\n{train['label'].value_counts()}")
print(f"Nulls: {train.isnull().sum().to_dict()}")
print(f"Duplicates: {train.duplicated().sum()}")

print("\n=== VALIDATE ===")
print(f"Shape: {validate.shape}")
print(f"Label counts:\n{validate['label'].value_counts()}")


=== TRAIN ===
Shape: (2700, 2)
Columns: ['text', 'label']
Label counts:
label
Neutral         1166
Personal         856
Geopolitical     401
Religious        158
Political        119
Name: count, dtype: int64
Nulls: {'text': 0, 'label': 0}
Duplicates: 33

=== VALIDATE ===
Shape: (900, 2)
Label counts:
label
Neutral         433
Personal        276
Geopolitical    110
Religious        44
Political        37
Name: count, dtype: int64


In [ ]:
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"✅ Random seed set to {SEED}")

✅ Random seed set to 42


In [ ]:
import json

config = {
    "seed": 42,
    "base_path": BASE,
    "num_labels": 5,
    "label_classes": ["Geopolitical", "Neutral", "Personal", "Political", "Religious"],
    "model_name": "sagorsarker/bangla-bert-base",
    "max_length": 128,
    "batch_size": 16,
    "learning_rate": 2e-5,
    "epochs": 4,
    "train_size": 2700,
    "val_size": 450,
    "test_size": 450
}

with open(f'{BASE}/models/config.json', 'w') as f:
    json.dump(config, f, indent=4)

print("✅ Config saved to models/config.json")
print(json.dumps(config, indent=4))

✅ Config saved to models/config.json
{
    "seed": 42,
    "base_path": "/content/drive/MyDrive/BanglaFakeNews",
    "num_labels": 5,
    "label_classes": [
        "Geopolitical",
        "Neutral",
        "Personal",
        "Political",
        "Religious"
    ],
    "model_name": "sagorsarker/bangla-bert-base",
    "max_length": 128,
    "batch_size": 16,
    "learning_rate": 2e-05,
    "epochs": 4,
    "train_size": 2700,
    "val_size": 450,
    "test_size": 450
}
